# Semantic Chunking

In [2]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

In [3]:
# What is Semantic Chunking?
# ------------------------------------------------------------
# Semantic chunking splits text based on MEANING not size.
#
# How it works:
# 1. Convert each sentence to a vector (meaning as numbers)
# 2. Compare similarity between neighboring sentences
# 3. When similarity DROPS suddenly = topic changed = split here
#
# Example:
# Sentence about admission → Sentence about admission = HIGH similarity
# Sentence about admission → Sentence about scholarship = LOW similarity
# LOW similarity = PUT A CHUNK BOUNDARY HERE
#
# Based on: Allamraju et al. (2025) - Paper 5 in our research
#
# ADVANTAGE: Splits at meaningful topic changes
# DISADVANTAGE: Needs embedding model — slower than fixed-size
# ------------------------------------------------------------

In [4]:
# Sample text
sample_text = """
Students who wish to apply for the Bachelor of Information and 
Communication Technology degree must meet the following requirements. 
The applicant must have completed Advanced Level examination with 
passes in three subjects. Mathematics must be one of the three 
subjects passed. The minimum score required is 200 marks combined 
across all three subjects.
The university offers merit-based scholarships to outstanding students. 
To be eligible for the scholarship, a student must maintain a Grade 
Point Average of 3.5 or above throughout the academic year. The 
scholarship covers full tuition fees for one academic year. It also 
includes a monthly allowance of five thousand rupees. Students who 
receive disciplinary warnings are not eligible to apply.
All students must follow the examination rules strictly. Students 
must arrive at the examination hall at least fifteen minutes before 
the scheduled start time. Mobile phones and electronic devices are 
strictly prohibited inside the examination hall. Students caught 
cheating will face immediate disqualification. The examination 
results will be published within four weeks after the examination date.
"""

In [6]:
# Load embedding model
# We use a free open-source model from HuggingFace
# This converts text to vectors for similarity comparison
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    # Small model — fast — good for demonstration
)
print("Model loaded successfully!")

Loading embedding model...
Model loaded successfully!


In [7]:
# Create semantic chunker
# breakpoint_threshold_type="percentile" means:
# Split when similarity drops below a certain percentile
chunker = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=70
    # 70 = split when similarity is in bottom 30%
)

In [8]:
# Apply chunking
print("Applying semantic chunking...")
chunks = chunker.split_text(sample_text)

Applying semantic chunking...


In [9]:
# Show results
print("=" * 60)
print("SEMANTIC CHUNKING RESULTS")
print("=" * 60)
print(f"Total chunks created: {len(chunks)}")
print()

for i, chunk in enumerate(chunks):
    print(f"--- CHUNK {i+1} ---")
    print(chunk)
    print(f"[Length: {len(chunk)} characters]")
    print()

SEMANTIC CHUNKING RESULTS
Total chunks created: 5

--- CHUNK 1 ---

Students who wish to apply for the Bachelor of Information and 
Communication Technology degree must meet the following requirements. The applicant must have completed Advanced Level examination with 
passes in three subjects.
[Length: 227 characters]

--- CHUNK 2 ---
Mathematics must be one of the three 
subjects passed. The minimum score required is 200 marks combined 
across all three subjects. The university offers merit-based scholarships to outstanding students. To be eligible for the scholarship, a student must maintain a Grade 
Point Average of 3.5 or above throughout the academic year. The 
scholarship covers full tuition fees for one academic year. It also 
includes a monthly allowance of five thousand rupees.
[Length: 460 characters]

--- CHUNK 3 ---
Students who 
receive disciplinary warnings are not eligible to apply.
[Length: 70 characters]

--- CHUNK 4 ---
All students must follow the examination rules s

In [10]:
# Explain what happened
print("=" * 60)
print("WHAT SEMANTIC CHUNKING DID")
print("=" * 60)
print("""
Notice how the chunks correspond to TOPICS:
- Chunk about admission requirements
- Chunk about scholarship eligibility  
- Chunk about examination rules

The AI detected when the topic changed
and placed the boundary exactly there.

This is much smarter than counting characters.
The chunker understood MEANING — not just size.
""")

WHAT SEMANTIC CHUNKING DID

Notice how the chunks correspond to TOPICS:
- Chunk about admission requirements
- Chunk about scholarship eligibility  
- Chunk about examination rules

The AI detected when the topic changed
and placed the boundary exactly there.

This is much smarter than counting characters.
The chunker understood MEANING — not just size.

